# Correcciones: DM con h=5 y bootstrap de clasificación

Aplica dos correcciones sobre los resultados del notebook principal (`anexo_codigo.ipynb`) sin reejecutar todo el pipeline:

1. **DM a 5 días con `h=5`:** regenera las predicciones a cinco días de Naïve, RF, XGBoost y LSTM con el protocolo exacto del anexo y recalcula el test con la ventana HAC adecuada para targets solapados.
2. **IC bootstrap de clasificación:** reconstruye las matrices de confusión desde `metricas_clasificacion_v4.csv` y aplica bootstrap multinomial (B=1000, semilla 42), sin reentrenar.

Protocolo idéntico al anexo: mismos datos, features, partición, features seleccionadas (de `features_seleccionadas_v4.csv`), grids, walk-forward (refit cada 60) y semillas. ARIMA a 5 días no se reejecuta (su DM ya usaba h=5); el bloque a 1 día tampoco (sin solapamiento, h=1 es correcto). Si existe `predicciones_5d_rerun.csv` de una ejecución previa, la celda de reanudación de la sección 5 evita repetir los grids de RF/XGBoost.

## 1. Librerías, rutas y constantes (idénticas al anexo)

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3})
sns.set_palette('tab10')
pd.set_option('display.float_format', '{:.4f}'.format)

print(f'pandas   {pd.__version__}')
print(f'numpy    {np.__version__}')
print(f'yfinance {yf.__version__}')

pandas   3.0.1
numpy    2.4.4
yfinance 1.2.0


In [2]:
from pathlib import Path

# Directorio raíz del proyecto.
BASE_DIR = Path(r'C:\Users\enris\Desktop\UNIVERSIDAD\Cuarto\TFG')

DIR_DATOS   = BASE_DIR / 'Datos'
DIR_STOCKS  = DIR_DATOS / 'Stocks'
DIR_BTC     = DIR_DATOS / 'BTC'
RESULTS_DIR = BASE_DIR / 'Resultados'
FIGURES_DIR = BASE_DIR / 'Figuras'

DIR_STOCKS.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Raíz del proyecto:', BASE_DIR)

Raíz del proyecto: C:\Users\enris\Desktop\UNIVERSIDAD\Cuarto\TFG


In [3]:
import itertools
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, accuracy_score,
                             precision_score, recall_score, f1_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import (RandomForestRegressor,
                              RandomForestClassifier)
from sklearn.inspection import permutation_importance

import xgboost as xgb

import statsmodels.api as sm
from pmdarima import auto_arima

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import (EarlyStopping,
                                        ReduceLROnPlateau)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

try:
    from dieboldmariano import dm_test
    DM_AVAILABLE = True
except ImportError:
    DM_AVAILABLE = False
    print("Aviso: 'dieboldmariano' no instalado.")

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'XGBoost:    {xgb.__version__}')
print(f'DM test:    '
      f'{"disponible" if DM_AVAILABLE else "NO instalado"}')

TensorFlow: 2.21.0
XGBoost:    3.2.0
DM test:    disponible


In [4]:
# Helper de progreso persistente en el notebook
def progress_iter(iterable, desc='', every=100, total=None):
    if total is None:
        try:
            total = len(iterable)
        except TypeError:
            total = None
    for i, x in enumerate(iterable):
        if total is not None:
            last = (i == total - 1)
            if i == 0 or (i + 1) % every == 0 or last:
                pct = 100.0 * (i + 1) / total
                print(f'  {desc}: {i+1}/{total} ({pct:.1f}%)')
        else:
            if (i + 1) % every == 0:
                print(f'  {desc}: {i+1} iter')
        yield x

In [5]:
STOCK_TICKERS = ['AAPL', 'JPM', 'JNJ', 'XOM']
ALL_TICKERS   = STOCK_TICKERS + ['BTC']

TRAIN_END     = '2022-12-31'
VAL_END       = '2023-12-31'

RETRAIN_EVERY = 60
SEQ_LEN       = 10

STOCKS_DIR = BASE_DIR / 'Datos/Stocks'
BTC_DIR    = BASE_DIR / 'Datos/BTC'

## 2. Carga de datos, VIX y features (idénticas al anexo)

In [6]:
def load_stock(ticker: str) -> pd.DataFrame:
    path = STOCKS_DIR / f'{ticker}_2015_2026.csv'
    df = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
    df.columns = [c.lower() for c in df.columns]
    df = df.rename(columns={'close':'Close','open':'Open',
                            'high':'High','low':'Low',
                            'volume':'Volume'})
    return df[['Open','High','Low','Close','Volume']].sort_index()


def load_btc(timeframe='1d') -> pd.DataFrame:
    fname = {'1d' :'btc_1d_data_2018_to_2025.csv',
             '4h' :'btc_4h_data_2018_to_2025.csv',
             '1h' :'btc_1h_data_2018_to_2025.csv',
             '15m':'btc_15m_data_2018_to_2025.csv'}
    path = BTC_DIR / fname[timeframe]
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(
        df['Open time'].str.replace(' UTC','', regex=False))
    df = df.set_index('Date').sort_index()
    cols = ['Open','High','Low','Close','Volume']
    return df[cols].astype(float)


raw = {t: load_stock(t) for t in STOCK_TICKERS}
raw['BTC'] = load_btc('1d')
for t in ALL_TICKERS:
    df = raw[t]
    print(f'{t:5s}: {len(df):,} sesiones  |  '
          f'{df.index[0].date()} — {df.index[-1].date()}')

AAPL : 2,815 sesiones  |  2015-01-02 — 2026-03-13
JPM  : 2,815 sesiones  |  2015-01-02 — 2026-03-13
JNJ  : 2,815 sesiones  |  2015-01-02 — 2026-03-13
XOM  : 2,815 sesiones  |  2015-01-02 — 2026-03-13
BTC  : 2,997 sesiones  |  2018-01-01 — 2026-03-16


In [7]:
# Descarga del índice VIX desde Yahoo Finance
print('Descargando datos del índice VIX...')
vix_raw = yf.download('^VIX', start='2014-01-01',
                      end='2026-06-01', progress=False)
if isinstance(vix_raw.columns, pd.MultiIndex):
    vix_raw.columns = vix_raw.columns.get_level_values(0)
vix_raw = vix_raw[['Close']].rename(columns={'Close':'VIX_Close'})
vix_raw.index = pd.to_datetime(vix_raw.index)

vix_raw['VIX_Return_1d'] = vix_raw['VIX_Close'].pct_change()
vix_raw['VIX_SMA_20']    = vix_raw['VIX_Close'].rolling(20).mean()
vix_raw['VIX_vs_SMA20']  = (vix_raw['VIX_Close']
                            / vix_raw['VIX_SMA_20'] - 1)

VIX_FEATURES = ['VIX_Close','VIX_Return_1d',
                'VIX_SMA_20','VIX_vs_SMA20']

print(f'VIX: {len(vix_raw):,} sesiones  |  '
      f'{vix_raw.index[0].date()} — {vix_raw.index[-1].date()}')

Descargando datos del índice VIX...
VIX: 3,121 sesiones  |  2014-01-02 — 2026-05-29


In [8]:
def add_features_v3(df: pd.DataFrame, vix_df=None):
    '''Bloque base + regimen de volatilidad + VIX (opcional).'''
    d = df.copy()

    # ── Bloque 1: indicadores base ─────────────────────────
    d['Return_1d']  = d['Close'].pct_change(1)
    d['Return_5d']  = d['Close'].pct_change(5)
    d['Return_10d'] = d['Close'].pct_change(10)
    d['Return_20d'] = d['Close'].pct_change(20)

    for w in [20, 50, 200]:
        d[f'SMA_{w}'] = d['Close'].rolling(w).mean()
    ema12 = d['Close'].ewm(span=12, adjust=False).mean()
    ema26 = d['Close'].ewm(span=26, adjust=False).mean()

    d['Price_vs_SMA20']  = d['Close'] / d['SMA_20']  - 1
    d['Price_vs_SMA50']  = d['Close'] / d['SMA_50']  - 1
    d['Price_vs_SMA200'] = d['Close'] / d['SMA_200'] - 1

    d['MACD']        = ema12 - ema26
    d['MACD_Signal'] = d['MACD'].ewm(span=9, adjust=False).mean()
    d['MACD_Hist']   = d['MACD'] - d['MACD_Signal']

    delta = d['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['RSI_14'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    bb_mid = d['Close'].rolling(20).mean()
    bb_std = d['Close'].rolling(20).std()
    bb_up  = bb_mid + 2 * bb_std
    bb_lo  = bb_mid - 2 * bb_std
    d['BB_Width']    = (bb_up - bb_lo) / bb_mid
    d['BB_Position'] = (d['Close'] - bb_lo) / (bb_up - bb_lo + 1e-8)

    tr = pd.concat([
        d['High'] - d['Low'],
        (d['High'] - d['Close'].shift(1)).abs(),
        (d['Low']  - d['Close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    d['ATR_rel'] = tr.ewm(com=13, adjust=False).mean() / d['Close']

    d['Momentum_5']  = d['Close'] - d['Close'].shift(5)
    d['Momentum_10'] = d['Close'] - d['Close'].shift(10)

    d['Vol_20d']   = d['Return_1d'].rolling(20).std()
    d['Vol_rel']   = (d['Volume']
                      / (d['Volume'].rolling(20).mean() + 1e-8))
    d['Day_Range'] = (d['High'] - d['Low']) / d['Close']

    # ── Bloque 2: régimen de volatilidad ───────────────────
    vol5  = d['Return_1d'].rolling(5).std()
    vol10 = d['Return_1d'].rolling(10).std()
    vol60 = d['Return_1d'].rolling(60).std()
    d['Vol_ratio_5_20']  = vol5  / (d['Vol_20d'] + 1e-8)
    d['Vol_ratio_10_60'] = vol10 / (vol60 + 1e-8)

    d['Vol_percentile_60'] = d['Vol_20d'].rolling(60).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)

    vol20_m = d['Vol_20d'].rolling(60).mean()
    vol20_s = d['Vol_20d'].rolling(60).std()
    d['Vol_zscore_60'] = (d['Vol_20d'] - vol20_m) / (vol20_s + 1e-8)

    log_hl = np.log(d['High'] / d['Low'])
    log_co = np.log(d['Close'] / d['Open'])
    gk = 0.5 * log_hl**2 - (2*np.log(2) - 1) * log_co**2
    d['Garman_Klass_vol'] = np.sqrt(gk.rolling(20).mean() * 252)

    d['Parkinson_vol'] = np.sqrt(
        (1 / (4*np.log(2)))
        * (np.log(d['High']/d['Low'])**2).rolling(20).mean() * 252)

    d['Vol_of_vol'] = d['Vol_20d'].rolling(20).std()

    vol_rel_60 = (d['Vol_20d']
                  / (d['Vol_20d'].rolling(60).mean() + 1e-8))
    d['Volume_vol_ratio'] = d['Vol_rel'] / (vol_rel_60 + 1e-8)

    # ── Bloque 3: VIX (solo acciones) ──────────────────────
    if vix_df is not None:
        d = d.join(vix_df[VIX_FEATURES], how='left')
        for col in VIX_FEATURES:
            d[col] = d[col].ffill()

    for c in ['SMA_20','SMA_50','SMA_200']:
        d.drop(columns=[c], errors='ignore', inplace=True)

    return d


FEATURE_COLS_STOCKS = [
    'Return_1d','Return_5d','Return_10d','Return_20d',
    'Price_vs_SMA20','Price_vs_SMA50','Price_vs_SMA200',
    'MACD','MACD_Signal','MACD_Hist','RSI_14',
    'BB_Width','BB_Position','ATR_rel',
    'Momentum_5','Momentum_10','Vol_20d','Vol_rel','Day_Range',
    'Vol_ratio_5_20','Vol_ratio_10_60','Vol_percentile_60',
    'Vol_zscore_60','Garman_Klass_vol','Parkinson_vol',
    'Vol_of_vol','Volume_vol_ratio',
] + VIX_FEATURES

FEATURE_COLS_BTC = [c for c in FEATURE_COLS_STOCKS
                    if c not in VIX_FEATURES]

data = {}
for t in STOCK_TICKERS:
    data[t] = add_features_v3(raw[t], vix_df=vix_raw)
data['BTC'] = add_features_v3(raw['BTC'], vix_df=None)

print(f'Features acciones: {len(FEATURE_COLS_STOCKS)}  |  '
      f'Features BTC: {len(FEATURE_COLS_BTC)}')

Features acciones: 31  |  Features BTC: 27


In [9]:
def add_weekly_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    logret = np.log(d['Close'] / d['Close'].shift(1))
    d['Return_5d_lag1'] = logret.rolling(5).sum().shift(1)
    d['Vol_5d']         = logret.rolling(5).std()
    d['Momentum_5d']    = d['Close'].pct_change(5)
    delta = d['Close'].diff()
    gain  = delta.clip(lower=0).rolling(5).mean()
    loss  = (-delta.clip(upper=0)).rolling(5).mean()
    d['RSI_5d'] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))
    d['Vol_mean_5d'] = (d['Volume'].rolling(5).mean()
                        / (d['Volume'].rolling(20).mean() + 1e-8))
    return d


WEEKLY_FEATURES = ['Return_5d_lag1', 'Vol_5d', 'Momentum_5d',
                   'RSI_5d', 'Vol_mean_5d']
print(f'Features semanales: {WEEKLY_FEATURES}')

Features semanales: ['Return_5d_lag1', 'Vol_5d', 'Momentum_5d', 'RSI_5d', 'Vol_mean_5d']


In [10]:
def prepare_dataset(df: pd.DataFrame, feature_cols: list):
    d = df.copy()
    d['logret']     = np.log(d['Close'] / d['Close'].shift(1))
    d['target_1d']  = d['logret'].shift(-1)
    d['target_5d']  = np.log(d['Close'].shift(-5) / d['Close'])
    d['target_cls'] = (d['target_1d'] > 0).astype(int)

    d = add_weekly_features(d)

    feats_1d = [c for c in feature_cols if c in d.columns]
    feats_5d = feats_1d + [c for c in WEEKLY_FEATURES
                           if c in d.columns]

    all_cols = list(set(feats_5d + ['target_1d','target_5d']))
    d = d.dropna(subset=all_cols)

    train = d.loc[:TRAIN_END]
    val   = d.loc[TRAIN_END:VAL_END].iloc[1:]
    test  = d.loc[VAL_END:].iloc[1:]

    return {
        'X_train_1d': train[feats_1d],
        'X_val_1d'  : val[feats_1d],
        'X_test_1d' : test[feats_1d],
        'X_train_5d': train[feats_5d],
        'X_val_5d'  : val[feats_5d],
        'X_test_5d' : test[feats_5d],
        'y_train_1d': train['target_1d'],
        'y_val_1d'  : val['target_1d'],
        'y_test_1d' : test['target_1d'],
        'y_train_5d': train['target_5d'],
        'y_val_5d'  : val['target_5d'],
        'y_test_5d' : test['target_5d'],
        'y_train_cls': train['target_cls'],
        'y_val_cls'  : val['target_cls'],
        'y_test_cls' : test['target_cls'],
        'logret_full': d['logret'],
        'feature_cols_1d': feats_1d,
        'feature_cols_5d': feats_5d,
    }


datasets = {}
for t in STOCK_TICKERS:
    datasets[t] = prepare_dataset(data[t], FEATURE_COLS_STOCKS)
datasets['BTC'] = prepare_dataset(data['BTC'], FEATURE_COLS_BTC)

print(f"{'Ticker':<8} {'Train':>6} {'Val':>6} {'Test':>6}  "
      f"{'Feat 1d':>8} {'Feat 5d':>8}")
print('-' * 55)
for t in ALL_TICKERS:
    ds = datasets[t]
    print(f"{t:<8} {len(ds['X_train_1d']):>6,} "
          f"{len(ds['X_val_1d']):>6,} "
          f"{len(ds['X_test_1d']):>6,}  "
          f"{len(ds['feature_cols_1d']):>8} "
          f"{len(ds['feature_cols_5d']):>8}")

Ticker    Train    Val   Test   Feat 1d  Feat 5d
-------------------------------------------------------
AAPL      1,732    245    517        31       36
JPM       1,745    231    512        31       36
JNJ       1,750    237    511        31       36
XOM       1,753    237    521        31       36
BTC       1,577    356    774        27       32


In [11]:
def directional_accuracy(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return 0.0
    return np.mean(
        np.sign(y_pred[mask]) == np.sign(y_true[mask]))


def bootstrap_ci(y_true, y_pred, metric_fn,
                   n_boot=1000, ci=0.95):
    rng = np.random.default_rng(42)
    n   = len(y_true)
    vals = [metric_fn(y_true[idx], y_pred[idx])
            for idx in (rng.integers(0, n, n)
                        for _ in range(n_boot))]
    alpha = 1 - ci
    return (np.percentile(vals, 100*alpha/2),
            np.percentile(vals, 100*(1-alpha/2)))


def evaluate_regression(y_true, y_pred, model_name, ticker,
                          horizon='1d'):
    yt = np.asarray(y_true, float)
    yp = np.asarray(y_pred, float)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    da   = directional_accuracy(yt, yp)

    rmse_ci = bootstrap_ci(
        yt, yp,
        lambda a, b: np.sqrt(mean_squared_error(a, b)))
    da_ci = bootstrap_ci(yt, yp, directional_accuracy)

    result = {
        'Model': model_name, 'Ticker': ticker,
        'Horizon': horizon,
        'RMSE': rmse, 'RMSE_CI_low': rmse_ci[0],
        'RMSE_CI_high': rmse_ci[1],
        'MAE': mae, 'R2': r2,
        'DA': da, 'DA_CI_low': da_ci[0],
        'DA_CI_high': da_ci[1]}

    print(f"{ticker:5s} {model_name:15s} {horizon:3s}  "
          f"RMSE: {rmse:.6f} "
          f"[{rmse_ci[0]:.6f}, {rmse_ci[1]:.6f}]  "
          f"DA: {da:.3f} [{da_ci[0]:.3f}, {da_ci[1]:.3f}]")
    return result


def evaluate_classification(y_true, y_pred, model_name, ticker):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    result = {'Model': model_name, 'Ticker': ticker,
              'Accuracy': acc, 'Precision': prec,
              'Recall': rec, 'F1': f1}
    print(f"{ticker:5s} {model_name:15s}  "
          f"Acc: {acc:.3f}  Prec: {prec:.3f}  "
          f"Rec: {rec:.3f}  F1: {f1:.3f}")
    return result

## 3. Features seleccionadas

Cargadas del CSV del anexo (mismas variables, sin recomputar).

In [12]:
feats_df = pd.read_csv(
    RESULTS_DIR / 'features_seleccionadas_v4.csv')

selected_features_rf  = {}
selected_features_xgb = {}
for (t, hz, grp), g in feats_df.groupby(
        ['Ticker', 'Horizon', 'Model_group']):
    feats = list(g.sort_values('Rank')['Feature'])
    if grp == 'RF':
        selected_features_rf[(t, hz)] = feats
    else:
        selected_features_xgb[(t, hz)] = feats

for t in ALL_TICKERS:
    n_rf  = len(selected_features_rf.get((t, '5d'), []))
    n_xgb = len(selected_features_xgb.get((t, '5d'), []))
    print(f'{t:5s} 5d: RF {n_rf} features | '
          f'XGB/LSTM {n_xgb} features')


AAPL  5d: RF 11 features | XGB/LSTM 13 features
JPM   5d: RF 8 features | XGB/LSTM 8 features
JNJ   5d: RF 8 features | XGB/LSTM 8 features
XOM   5d: RF 7 features | XGB/LSTM 9 features
BTC   5d: RF 13 features | XGB/LSTM 8 features


## 4. Modelos (definiciones idénticas al anexo)

In [13]:
def run_rf(ticker, ds, horizon='1d',
           retrain_every=RETRAIN_EVERY, selected_feats=None, 
           progress_every=100):
    feat_key = f'feature_cols_{horizon}'
    all_feats = list(ds[feat_key])
    if selected_feats is not None:
        feat_idx = [all_feats.index(f) for f in selected_feats
                    if f in all_feats]
    else:
        feat_idx = list(range(len(all_feats)))

    X_tr = ds[f'X_train_{horizon}'].values[:, feat_idx]
    X_vl = ds[f'X_val_{horizon}'].values[:, feat_idx]
    X_te = ds[f'X_test_{horizon}'].values[:, feat_idx]
    y_tr = ds[f'y_train_{horizon}'].values
    y_vl = ds[f'y_val_{horizon}'].values
    y_te = ds[f'y_test_{horizon}'].values

    scaler_s = StandardScaler()
    X_search = scaler_s.fit_transform(np.vstack([X_tr, X_vl]))
    y_search = np.concatenate([y_tr, y_vl])
    tscv = TimeSeriesSplit(n_splits=5)
    param_grid = [{'n_estimators': n, 'max_depth': d,
                   'min_samples_split': s}
                  for n in [100, 200, 300]
                  for d in [5, 10, 15]
                  for s in [5, 10, 20]]
    best_score, best_params = np.inf, {}
    print(f"  {ticker} ({horizon}): búsqueda RF "
          f"({len(param_grid)} combs, "
          f"{len(feat_idx)} features)...")
    for params in param_grid:
        scores = []
        for tr_i, vl_i in tscv.split(X_search):
            rf = RandomForestRegressor(
                **params, n_jobs=-1, random_state=42)
            rf.fit(X_search[tr_i], y_search[tr_i])
            scores.append(np.sqrt(mean_squared_error(
                y_search[vl_i], rf.predict(X_search[vl_i]))))
        if np.mean(scores) < best_score:
            best_score, best_params = np.mean(scores), params
    print(f"  {ticker} ({horizon}): mejor = {best_params}")

    X_pool = np.vstack([X_tr, X_vl])
    y_pool = np.concatenate([y_tr, y_vl])
    preds = []
    for i in progress_iter(range(len(y_te)),
                           desc=f'  {ticker} RF {horizon}',
                           every=progress_every):
        if i == 0 or i % retrain_every == 0:
            sc = StandardScaler()
            end = (len(X_pool) - len(y_te) + i
                   if i > 0 else len(X_pool))
            X_fit = sc.fit_transform(X_pool[:end])
            y_fit = y_pool[:end]
            rf = RandomForestRegressor(
                **best_params, n_jobs=-1, random_state=42)
            rf.fit(X_fit, y_fit)
        preds.append(rf.predict(sc.transform(X_te[i:i+1]))[0])
    return np.array(preds)


In [14]:
def run_xgboost(ticker, ds, horizon='1d',
                 retrain_every=RETRAIN_EVERY,
                 selected_feats=None,
                 progress_every=100):
    feat_key = f'feature_cols_{horizon}'
    all_feats = list(ds[feat_key])
    if selected_feats is not None:
        feat_idx = [all_feats.index(f) for f in selected_feats
                    if f in all_feats]
    else:
        feat_idx = list(range(len(all_feats)))

    X_tr = ds[f'X_train_{horizon}'].values[:, feat_idx]
    X_vl = ds[f'X_val_{horizon}'].values[:, feat_idx]
    X_te = ds[f'X_test_{horizon}'].values[:, feat_idx]
    y_tr = ds[f'y_train_{horizon}'].values
    y_vl = ds[f'y_val_{horizon}'].values
    y_te = ds[f'y_test_{horizon}'].values

    scaler_s = StandardScaler()
    X_search = scaler_s.fit_transform(np.vstack([X_tr, X_vl]))
    y_search = np.concatenate([y_tr, y_vl])
    tscv = TimeSeriesSplit(n_splits=5)
    param_grid = [
        {'n_estimators': n, 'max_depth': d,
         'learning_rate': lr, 'subsample': ss,
         'colsample_bytree': cs, 'gamma': g,
         'reg_alpha': ra, 'reg_lambda': rl}
        for n in [100, 200] for d in [3, 5]
        for lr in [0.01, 0.05] for ss in [0.7, 0.8]
        for cs in [0.7, 1.0]
        for g in [0, 0.1] for ra in [0, 0.1]
        for rl in [1, 3]]
    best_score, best_params = np.inf, {}
    print(f"  {ticker} ({horizon}): búsqueda XGBoost "
          f"({len(param_grid)} combs, "
          f"{len(feat_idx)} features)...")
    for params in param_grid:
        scores = []
        for tr_i, vl_i in tscv.split(X_search):
            mdl = xgb.XGBRegressor(
                **params, random_state=42, verbosity=0)
            mdl.fit(X_search[tr_i], y_search[tr_i])
            scores.append(np.sqrt(mean_squared_error(
                y_search[vl_i], mdl.predict(X_search[vl_i]))))
        if np.mean(scores) < best_score:
            best_score, best_params = np.mean(scores), params
    print(f"  {ticker} ({horizon}): mejor = {best_params}")

    X_pool = np.vstack([X_tr, X_vl])
    y_pool = np.concatenate([y_tr, y_vl])
    preds = []
    for i in progress_iter(range(len(y_te)),
                           desc=f'  {ticker} XGB {horizon}',
                           every=progress_every):
        if i == 0 or i % retrain_every == 0:
            sc = StandardScaler()
            end = (len(X_pool) - len(y_te) + i
                   if i > 0 else len(X_pool))
            X_fit = sc.fit_transform(X_pool[:end])
            y_fit = y_pool[:end]
            mdl = xgb.XGBRegressor(
                **best_params, random_state=42, verbosity=0)
            mdl.fit(X_fit, y_fit)
        preds.append(mdl.predict(sc.transform(X_te[i:i+1]))[0])
    return np.array(preds)


In [15]:
def build_lstm(seq_len, n_features):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        LSTM(16, return_sequences=False),
        Dropout(0.3),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss='mse', metrics=['mae'])
    return model


def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len + 1):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len-1])
    return np.array(Xs), np.array(ys)


# Historial de entrenamiento de la primera LSTM ajustada
lstm_histories = {}


def run_lstm(ticker, ds, horizon='1d', seq_len=SEQ_LEN,
             retrain_every=RETRAIN_EVERY, epochs=100,
             batch_size=32):
    X_tr = ds['X_train'].values
    X_vl = ds['X_val'].values
    X_te = ds['X_test'].values
    y_tr = ds[f'y_train_{horizon}'].values
    y_vl = ds[f'y_val_{horizon}'].values
    y_te = ds[f'y_test_{horizon}'].values
    n_feat = X_tr.shape[1]

    sc_X = StandardScaler()
    sc_y = StandardScaler()
    X_pool = sc_X.fit_transform(np.vstack([X_tr, X_vl]))
    X_te_sc = sc_X.transform(X_te)
    y_pool = np.concatenate([y_tr, y_vl])
    y_pool_sc = sc_y.fit_transform(
        y_pool.reshape(-1, 1)).ravel()

    X_tr_seq, y_tr_seq = create_sequences(
        sc_X.transform(X_tr),
        sc_y.transform(y_tr.reshape(-1, 1)).ravel(), seq_len)
    X_vl_seq, y_vl_seq = create_sequences(
        X_pool[len(X_tr)-seq_len+1:len(X_tr)+len(X_vl)],
        y_pool_sc[len(X_tr)-seq_len+1:len(X_tr)+len(X_vl)],
        seq_len)

    model = build_lstm(seq_len, n_feat)
    cb = [EarlyStopping(patience=10,
                        restore_best_weights=True,
                        monitor='val_loss'),
          ReduceLROnPlateau(factor=0.5, patience=5,
                             min_lr=1e-6, verbose=0)]
    hist = model.fit(
        X_tr_seq, y_tr_seq,
        validation_data=(X_vl_seq, y_vl_seq),
        epochs=epochs, batch_size=batch_size,
        callbacks=cb, verbose=0)
    lstm_histories[(ticker, horizon)] = hist.history
    print(f"  {ticker} ({horizon}): entrenado "
          f"{len(hist.history['loss'])} épocas")

    full_X = np.vstack([X_pool, X_te_sc])
    pool_len = len(X_pool)
    preds = []
    for i in progress_iter(range(len(y_te)),
                           desc=f'  {ticker} LSTM {horizon}',
                           every=100):
        start = max(0, pool_len + i - seq_len + 1)
        x_seq = full_X[start:pool_len+i+1]
        if len(x_seq) < seq_len:
            x_seq = np.pad(
                x_seq, ((seq_len-len(x_seq), 0), (0, 0)),
                mode='edge')
        pred_sc = model.predict(
            x_seq[-seq_len:].reshape(1, seq_len, n_feat),
            verbose=0)[0, 0]
        preds.append(
            sc_y.inverse_transform([[pred_sc]])[0, 0])

        if (i+1) % retrain_every == 0 and i < len(y_te)-1:
            y_obs = np.concatenate([
                y_pool_sc,
                sc_y.transform(
                    y_te[:i+1].reshape(-1, 1)).ravel()])
            X_rt, y_rt = create_sequences(
                full_X[:pool_len+i+1], y_obs, seq_len)
            model.fit(X_rt, y_rt, epochs=10,
                      batch_size=batch_size, verbose=0)
    return np.array(preds)


## 5. Ejecución a 5 días

Dos caminos. En la primera ejecución corro las celdas en orden: Naïve → RF → XGBoost → LSTM. Si reinicio el kernel y ya existe `Resultados/predicciones_5d_rerun.csv` de una corrida anterior, uso la celda de reanudación y después la de la LSTM, sin repetir Naïve/RF/XGBoost.

In [16]:
# === REANUDACION tras reinicio de kernel ===
# Recupera las predicciones ya calculadas (Naive/RF/XGBoost)
# del CSV exportado en la seccion 6 de una ejecucion previa.
# Tras esta celda, salta DIRECTAMENTE a la celda de la LSTM.
HORIZON = '5d'
all_results_reg = []
all_preds_reg   = {}

prev = pd.read_csv(RESULTS_DIR / 'predicciones_5d_rerun.csv')
for (t, hz, m), g in prev.groupby(
        ['Ticker', 'Horizon', 'Model']):
    y_pred = g.sort_values('idx')['pred'].values
    y_test = datasets[t][f'y_test_{hz}'].values
    if len(y_pred) != len(y_test):
        print(f'AVISO: {t} {m} longitud {len(y_pred)} != '
              f'{len(y_test)}, se omite')
        continue
    all_preds_reg[(t, hz, m)] = y_pred
    all_results_reg.append(evaluate_regression(
        y_test, y_pred, m, t, hz))

print('Predicciones recuperadas:')
for t, m in sorted({(t, m) for (t, h, m) in all_preds_reg}):
    print(f'  {t:5s} {m}')


AAPL  Naïve           5d   RMSE: 0.040859 [0.036810, 0.045451]  DA: 0.000 [0.000, 0.000]
AAPL  Random Forest   5d   RMSE: 0.040987 [0.036629, 0.045728]  DA: 0.516 [0.474, 0.559]
AAPL  XGBoost         5d   RMSE: 0.040687 [0.036425, 0.045381]  DA: 0.540 [0.499, 0.580]
BTC   Naïve           5d   RMSE: 0.053596 [0.050593, 0.056718]  DA: 0.000 [0.000, 0.000]
BTC   Random Forest   5d   RMSE: 0.058960 [0.055874, 0.061955]  DA: 0.465 [0.432, 0.500]
BTC   XGBoost         5d   RMSE: 0.053928 [0.050919, 0.056997]  DA: 0.514 [0.481, 0.549]
JNJ   Naïve           5d   RMSE: 0.023902 [0.022206, 0.025547]  DA: 0.000 [0.000, 0.000]
JNJ   Random Forest   5d   RMSE: 0.024355 [0.022684, 0.026042]  DA: 0.483 [0.442, 0.525]
JNJ   XGBoost         5d   RMSE: 0.023674 [0.021985, 0.025276]  DA: 0.569 [0.530, 0.613]
JPM   Naïve           5d   RMSE: 0.034335 [0.031712, 0.037078]  DA: 0.000 [0.000, 0.000]
JPM   Random Forest   5d   RMSE: 0.037535 [0.034391, 0.040574]  DA: 0.574 [0.531, 0.619]
JPM   XGBoost        

In [17]:
HORIZON = '5d'
if 'all_preds_reg' not in dir():
    all_results_reg = []
    all_preds_reg   = {}

print(f'Naïve ({HORIZON})...')
for t in ALL_TICKERS:
    ds = datasets[t]
    y_test = ds[f'y_test_{HORIZON}'].values
    y_pred = np.zeros(len(y_test))
    result = evaluate_regression(
        y_test, y_pred, 'Naïve', t, HORIZON)
    all_results_reg.append(result)
    all_preds_reg[(t, HORIZON, 'Naïve')] = y_pred
print('Naïve listo.')


Naïve (5d)...
AAPL  Naïve           5d   RMSE: 0.040859 [0.036810, 0.045451]  DA: 0.000 [0.000, 0.000]
JPM   Naïve           5d   RMSE: 0.034335 [0.031712, 0.037078]  DA: 0.000 [0.000, 0.000]
JNJ   Naïve           5d   RMSE: 0.023902 [0.022206, 0.025547]  DA: 0.000 [0.000, 0.000]
XOM   Naïve           5d   RMSE: 0.032118 [0.029312, 0.034957]  DA: 0.000 [0.000, 0.000]
BTC   Naïve           5d   RMSE: 0.053596 [0.050593, 0.056718]  DA: 0.000 [0.000, 0.000]
Naïve listo.


In [18]:
print('Ejecutando Random Forest walk-forward (5d)...')
for t in ALL_TICKERS:
    ds = datasets[t]
    y_test = ds[f'y_test_{HORIZON}'].values
    rf_feats = selected_features_rf.get((t, HORIZON))
    try:
        y_pred = run_rf(t, ds, HORIZON,
                         selected_feats=rf_feats)
        result = evaluate_regression(
            y_test, y_pred, 'Random Forest', t, HORIZON)
        all_results_reg.append(result)
        all_preds_reg[(t, HORIZON, 'Random Forest')] = y_pred
    except Exception as e:
        print(f'  Error en RF {t} {HORIZON}: {e}')


Ejecutando Random Forest walk-forward (5d)...
  AAPL (5d): búsqueda RF (27 combs, 11 features)...


KeyboardInterrupt: 

In [ ]:
print('Ejecutando XGBoost walk-forward (5d)...')
for t in ALL_TICKERS:
    ds = datasets[t]
    y_test = ds[f'y_test_{HORIZON}'].values
    xgb_feats = selected_features_xgb.get((t, HORIZON))
    try:
        y_pred = run_xgboost(t, ds, HORIZON,
                              selected_feats=xgb_feats)
        result = evaluate_regression(
            y_test, y_pred, 'XGBoost', t, HORIZON)
        all_results_reg.append(result)
        all_preds_reg[(t, HORIZON, 'XGBoost')] = y_pred
    except Exception as e:
        print(f'  Error en XGBoost {t} {HORIZON}: {e}')


In [19]:
print('Ejecutando LSTM walk-forward (5d)...')
for t in ALL_TICKERS:
    ds = datasets[t]
    xgb_feats = selected_features_xgb.get((t, HORIZON))
    feat_key = f'feature_cols_{HORIZON}'
    all_feats = list(ds[feat_key])
    if xgb_feats is not None:
        feat_idx = [all_feats.index(f) for f in xgb_feats
                    if f in all_feats]
    else:
        feat_idx = list(range(len(all_feats)))

    ds_lstm = {
        'X_train': pd.DataFrame(
            ds[f'X_train_{HORIZON}'].values[:, feat_idx]),
        'X_val': pd.DataFrame(
            ds[f'X_val_{HORIZON}'].values[:, feat_idx]),
        'X_test': pd.DataFrame(
            ds[f'X_test_{HORIZON}'].values[:, feat_idx]),
        f'y_train_{HORIZON}': ds[f'y_train_{HORIZON}'],
        f'y_val_{HORIZON}':   ds[f'y_val_{HORIZON}'],
        f'y_test_{HORIZON}':  ds[f'y_test_{HORIZON}'],
        'feature_cols': list(range(len(feat_idx))),
    }
    y_test = ds[f'y_test_{HORIZON}'].values
    try:
        y_pred = run_lstm(t, ds_lstm, HORIZON)
        result = evaluate_regression(
            y_test, y_pred, 'LSTM', t, HORIZON)
        all_results_reg.append(result)
        all_preds_reg[(t, HORIZON, 'LSTM')] = y_pred
    except Exception as e:
        print(f'  Error en LSTM {t} {HORIZON}: {e}')


Ejecutando LSTM walk-forward (5d)...
  AAPL (5d): entrenado 13 épocas
    AAPL LSTM 5d: 1/517 (0.2%)
    AAPL LSTM 5d: 100/517 (19.3%)
    AAPL LSTM 5d: 200/517 (38.7%)
    AAPL LSTM 5d: 300/517 (58.0%)
    AAPL LSTM 5d: 400/517 (77.4%)
    AAPL LSTM 5d: 500/517 (96.7%)
    AAPL LSTM 5d: 517/517 (100.0%)
AAPL  LSTM            5d   RMSE: 0.042413 [0.038691, 0.046681]  DA: 0.536 [0.495, 0.580]
  JPM (5d): entrenado 13 épocas
    JPM LSTM 5d: 1/512 (0.2%)
    JPM LSTM 5d: 100/512 (19.5%)
    JPM LSTM 5d: 200/512 (39.1%)
    JPM LSTM 5d: 300/512 (58.6%)
    JPM LSTM 5d: 400/512 (78.1%)
    JPM LSTM 5d: 500/512 (97.7%)
    JPM LSTM 5d: 512/512 (100.0%)
JPM   LSTM            5d   RMSE: 0.036750 [0.033457, 0.040338]  DA: 0.555 [0.508, 0.600]
  JNJ (5d): entrenado 14 épocas
    JNJ LSTM 5d: 1/511 (0.2%)
    JNJ LSTM 5d: 100/511 (19.6%)
    JNJ LSTM 5d: 200/511 (39.1%)
    JNJ LSTM 5d: 300/511 (58.7%)
    JNJ LSTM 5d: 400/511 (78.3%)
    JNJ LSTM 5d: 500/511 (97.8%)
    JNJ LSTM 5d: 511/511 (10

## 6. Verificación de reproducibilidad

RF y XGBoost deben coincidir al decimal con
`metricas_regresion_v4.csv`; la LSTM puede desviarse ligeramente.
Reexporta las predicciones (ahora con la LSTM incluida).

In [20]:
ref = pd.read_csv(RESULTS_DIR / 'metricas_regresion_v4.csv')
ref5 = ref[ref['Horizon'] == '5d'].set_index(
    ['Ticker', 'Model'])

nuevo = pd.DataFrame(all_results_reg).set_index(
    ['Ticker', 'Model'])

print(f'{"Ticker":6s} {"Modelo":14s} {"RMSE nuevo":>11s} '
      f'{"RMSE anexo":>11s} {"delta %":>8s}')
print('-' * 56)
for (t, m), row in nuevo.iterrows():
    if (t, m) not in ref5.index:
        continue
    r_new = row['RMSE']
    r_old = ref5.loc[(t, m), 'RMSE']
    d = 100 * (r_new - r_old) / r_old
    flag = '  <-- revisar' if abs(d) > 2 else ''
    print(f'{t:6s} {m:14s} {r_new:11.6f} '
          f'{r_old:11.6f} {d:+8.3f}{flag}')

pred_rows = []
for (t, hz, m), pred in all_preds_reg.items():
    for i, v in enumerate(pred):
        pred_rows.append({'Ticker': t, 'Horizon': hz,
                          'Model': m, 'idx': i, 'pred': v})
pd.DataFrame(pred_rows).to_csv(
    RESULTS_DIR / 'predicciones_5d_rerun.csv', index=False)
print('\nPredicciones guardadas en '
      'Resultados/predicciones_5d_rerun.csv')


Ticker Modelo          RMSE nuevo  RMSE anexo  delta %
--------------------------------------------------------
AAPL   Naïve             0.040859    0.040859   +0.000
AAPL   Random Forest     0.040987    0.040987   +0.000
AAPL   XGBoost           0.040687    0.040687   +0.000
BTC    Naïve             0.053596    0.053596   +0.000
BTC    Random Forest     0.058960    0.058960   +0.000
BTC    XGBoost           0.053928    0.053928   +0.000
JNJ    Naïve             0.023902    0.023902   +0.000
JNJ    Random Forest     0.024355    0.024355   +0.000
JNJ    XGBoost           0.023674    0.023674   +0.000
JPM    Naïve             0.034335    0.034335   +0.000
JPM    Random Forest     0.037535    0.037535   +0.000
JPM    XGBoost           0.033996    0.033996   +0.000
XOM    Naïve             0.032118    0.032118   +0.000
XOM    Random Forest     0.034788    0.034788   +0.000
XOM    XGBoost           0.031930    0.031930   +0.000
AAPL   Naïve             0.040859    0.040859   +0.000
JPM    N

## 7. Test de Diebold-Mariano a 5 días con h=5

Con autodiagnóstico: comprueba el paquete y las predicciones
disponibles antes de calcular.

In [21]:
assert DM_AVAILABLE, (
    "El paquete 'dieboldmariano' no esta instalado: "
    "pip install dieboldmariano")

MODELS_DM = ['Naïve', 'Random Forest', 'XGBoost', 'LSTM']

faltan = [(t, m) for t in ALL_TICKERS for m in MODELS_DM
          if (t, '5d', m) not in all_preds_reg]
if faltan:
    print('AVISO: faltan predicciones de la seccion 5 para:')
    for t, m in faltan:
        print(f'  - {t} {m}')
    print('Los pares afectados se omiten del test.\n')

dm_old = pd.read_csv(RESULTS_DIR / 'diebold_mariano_v4.csv')
dm_old5 = dm_old[dm_old['Horizon'] == '5d']

dm_rows = []
print(f'{"Ticker":6s} {"Par":32s} {"DM h=5":>8s} '
      f'{"p h=5":>9s} {"p h=1":>9s} {"p anexo":>9s}')
print('-' * 80)
for t in ALL_TICKERS:
    y_true = datasets[t]['y_test_5d'].values
    for m1, m2 in itertools.combinations(MODELS_DM, 2):
        k1, k2 = (t, '5d', m1), (t, '5d', m2)
        if k1 not in all_preds_reg or k2 not in all_preds_reg:
            continue
        p1 = np.asarray(all_preds_reg[k1], dtype=float)
        p2 = np.asarray(all_preds_reg[k2], dtype=float)
        if len(p1) != len(y_true) or len(p2) != len(y_true):
            print(f'  {t} {m1}-{m2}: longitudes no cuadran, '
                  f'se omite')
            continue
        try:
            s5, pv5 = dm_test(y_true, p1, p2, h=5)
            s1, pv1 = dm_test(y_true, p1, p2, h=1)
        except Exception as e:
            print(f'  {t} {m1}-{m2}: error '
                  f'{type(e).__name__}: {e}')
            continue
        viejo = dm_old5[(dm_old5['Ticker'] == t)
                        & (dm_old5['Model1'] == m1)
                        & (dm_old5['Model2'] == m2)]
        p_old = (float(viejo['p_value'].iloc[0])
                 if len(viejo) else float('nan'))
        winner = m1 if s5 < 0 else m2
        dm_rows.append({
            'Ticker': t, 'Horizon': '5d',
            'Model1': m1, 'Model2': m2,
            'DM_h5': s5, 'p_h5': pv5,
            'DM_h1': s1, 'p_h1': pv1,
            'p_anexo_h1': p_old,
            'Winner_h5': winner if pv5 < 0.05 else ''})
        star = ' *' if pv5 < 0.05 else ''
        print(f'{t:6s} {m1+" vs "+m2:32s} {s5:+8.3f} '
              f'{pv5:9.4f} {pv1:9.4f} {p_old:9.4f}{star}')

if not dm_rows:
    print('\nNo se ha podido calcular ningun par.')
else:
    dm_df = pd.DataFrame(dm_rows)
    dm_df.to_csv(
        RESULTS_DIR / 'diebold_mariano_5d_h5.csv',
        index=False)
    print('\nGuardado en '
          'Resultados/diebold_mariano_5d_h5.csv')
    n5 = int((dm_df['p_h5'] < 0.05).sum())
    n1 = int((dm_df['p_h1'] < 0.05).sum())
    print(f'\nPares significativos con h=5: {n5} '
          f'(con h=1 en esta reejecucion: {n1})')


Ticker Par                                DM h=5     p h=5     p h=1   p anexo
--------------------------------------------------------------------------------
AAPL   Naïve vs Random Forest             -0.244    0.8075    0.6478       nan
AAPL   Naïve vs XGBoost                   +0.362    0.7173    0.4481       nan
AAPL   Naïve vs LSTM                      -1.348    0.1782    0.0050    0.0189
AAPL   Random Forest vs XGBoost           +0.640    0.5226    0.2802       nan
AAPL   Random Forest vs LSTM              -1.086    0.2782    0.0221    0.0364
AAPL   XGBoost vs LSTM                    -1.250    0.2117    0.0077    0.0168
JPM    Naïve vs Random Forest             -1.416    0.1575    0.0072    0.0072
JPM    Naïve vs XGBoost                   +1.106    0.2694    0.0176    0.0176
JPM    Naïve vs LSTM                      -1.427    0.1542    0.0038    0.0172
JPM    Random Forest vs XGBoost           +1.474    0.1411    0.0048    0.0048
JPM    Random Forest vs LSTM              +0.260  

## 8. IC bootstrap para la clasificación

Reconstrucción exacta de la matriz de confusión desde
`metricas_clasificacion_v4.csv` + bootstrap multinomial. (Si ya se
generó `metricas_clasificacion_ci_v4.csv` en una ejecución previa,
esta celda simplemente lo regenera con idéntico resultado.)

In [22]:
B = 1000
rng = np.random.default_rng(42)

cls = pd.read_csv(
    RESULTS_DIR / 'metricas_clasificacion_v4.csv')
n_map = {t: len(datasets[t]['y_test_1d'])
         for t in ALL_TICKERS}

def reconstruir(row, n, pos_rate):
    if 'Naïve' in row['Model']:
        tp = int(round(pos_rate * n))
        return tp, n - tp, 0, 0
    P  = int(round(pos_rate * n))
    tp = int(round(row['Recall'] * P))
    fp = int(round(tp / row['Precision'] - tp))
    fn = P - tp
    tn = n - P - fp
    return tp, fp, fn, tn

def metricas(tp, fp, fn, tn):
    n = tp + fp + fn + tn
    acc = (tp + tn) / n
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * prec * rec / (prec + rec)
          if prec + rec else 0.0)
    return acc, prec, rec, f1

rows = []
for t in ALL_TICKERS:
    n = n_map[t]
    sub = cls[cls['Ticker'] == t]
    pos_rate = sub[sub['Model'].str.contains('Naïve')][
        'Precision'].iloc[0]
    acc_base = sub[sub['Model'].str.contains('Naïve')][
        'Accuracy'].iloc[0]
    for _, row in sub.iterrows():
        tp, fp, fn, tn = reconstruir(row, n, pos_rate)
        acc, prec, rec, f1 = metricas(tp, fp, fn, tn)
        ok = (abs(acc - row['Accuracy']) < 2e-3
              and abs(f1 - row['F1']) < 5e-3)
        if not ok:
            print(f'AVISO {t} {row["Model"]}: matriz '
                  f'reconstruida no cuadra')
        probs = np.array([tp, fp, fn, tn]) / n
        sims = rng.multinomial(n, probs, size=B)
        boot = np.array([metricas(*s) for s in sims])
        lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
        rows.append({
            'Ticker': t, 'Model': row['Model'], 'n': n,
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Accuracy': row['Accuracy'],
            'Acc_CI_low': lo[0], 'Acc_CI_high': hi[0],
            'Delta_acc_vs_baseline':
                row['Accuracy'] - acc_base,
            'Precision': row['Precision'],
            'Prec_CI_low': lo[1], 'Prec_CI_high': hi[1],
            'Recall': row['Recall'],
            'Rec_CI_low': lo[2], 'Rec_CI_high': hi[2],
            'F1': row['F1'],
            'F1_CI_low': lo[3], 'F1_CI_high': hi[3]})

ci_df = pd.DataFrame(rows)
ci_df.to_csv(
    RESULTS_DIR / 'metricas_clasificacion_ci_v4.csv',
    index=False)
print('Guardado en '
      'Resultados/metricas_clasificacion_ci_v4.csv')


Guardado en Resultados/metricas_clasificacion_ci_v4.csv


## 9. Resultados y su uso en la memoria

Salidas del notebook y dónde alimentan el TFG:

1. **Tabla DM (bloque h=5):** valores `DM_h5`/`p_h5` de `diebold_mariano_5d_h5.csv`; en 5 días el test usa `h=5`.
2. **Tabla de clasificación:** columna de IC bootstrap desde `metricas_clasificacion_ci_v4.csv`.
3. **Metodología:** el DM a 5 días usa ventana HAC con `h=5` y corrección de Harvey. Si la LSTM se desvía en la verificación de la sección 6, sus filas de la tabla de regresión a 5 días se resincronizan con esta reejecución.

In [23]:
pd.DataFrame(all_results_reg).to_csv(
    RESULTS_DIR / 'metricas_regresion_5d_rerun.csv', index=False)

dm_df.to_csv(
    RESULTS_DIR / 'diebold_mariano_5d_h5.csv', index=False)

pred_rows = []
for (t, hz, m), pred in all_preds_reg.items():
    for i, v in enumerate(pred):
        pred_rows.append({'Ticker': t, 'Horizon': hz,
                          'Model': m, 'idx': i, 'pred': v})
pd.DataFrame(pred_rows).to_csv(
    RESULTS_DIR / 'predicciones_5d_rerun.csv', index=False)

print('Exportados:')
print(' - metricas_regresion_5d_rerun.csv')
print(' - diebold_mariano_5d_h5.csv')
print(' - predicciones_5d_rerun.csv')

Exportados:
 - metricas_regresion_5d_rerun.csv
 - diebold_mariano_5d_h5.csv
 - predicciones_5d_rerun.csv
